# 00. Подготовка: старые модели HW1 в HW2

Запускается **один раз**. Что делает:
1. переносит обученные модели HW1 в `hw2/artifacts/` под именами `hw1_<имя>.joblib` (+ копирует `dialog_flags.csv`);
2. собирает train/test и блоки кросс-валидации ровно как в HW1 (флаги `use_reaction`, группы `group_id`);
3. считает метрики старых моделей на кросс-валидации и записывает в `results/cv_table.csv`.

Дальше каждый в команде в своём ноутбуке запускает такую же ячейку подготовки и добавляет свою строку в ту же таблицу.

## 1. Перенос моделей HW1

`HW1_ARTIFACTS` — папка `artifacts/` из HW1 (там лежат `*_1.joblib` и `dialog_flags.csv`). Поправьте путь под себя.

Пояснение: `kirp_model` в HW1 сохранена с токенизатором из ноутбука, и сама по себе без ноутбука не загрузится.
`migrate_old_models` подменяет его на `solution_utils.smart_tokenizer` (копия вашей функции).

In [3]:
from pathlib import Path

import solution_utils as su

HW1_ARTIFACTS = Path("../solution_hw1_BKZ/artifacts")     # <-- поправьте, если папка HW1 лежит в другом месте
assert HW1_ARTIFACTS.exists(), f"нет папки {HW1_ARTIFACTS}: укажите, где лежит artifacts/ из HW1"

moved = su.migrate_old_models(HW1_ARTIFACTS)
print("перенесено:", list(moved))

перенесено: ['hw1_most_frequent', 'hw1_stratified', 'hw1_bow_logreg', 'hw1_tfidf_logreg', 'hw1_kirp_model']


## 2. Данные и разбиение

In [5]:
import numpy as np
import pandas as pd

import solution_utils as su
from solution_utils import CLASSES, SEED

# Задача 1 из HW1: диалоги с use_reaction == 1 (флаги из dialog_flags.csv), разбиение по group_id
task1 = su.load_task1(su.train_path(), su.flags_path())
train_df, test_df = su.split_task1(task1)          # то же train/test, что в HW1

train_dialogs = train_df[["dialog_id", "turns"]].to_dict("records")
test_dialogs = test_df[["dialog_id", "turns"]].to_dict("records")
y_train, y_test = train_df["reaction"].to_numpy(), test_df["reaction"].to_numpy()
folds = su.get_folds(y_train, train_df["group_id"].to_numpy())   # одни блоки CV для всех моделей

print(f"всего для задачи 1: {len(task1)} | train: {len(train_df)} | test: {len(test_df)}")
pd.DataFrame({"train": pd.Series(y_train).value_counts(normalize=True),
              "test": pd.Series(y_test).value_counts(normalize=True)}).loc[CLASSES].round(3)

AssertionError: не нашли hw1_train.jsonl: положите рядом или задайте HW2_TRAIN

Проверка: в HW1 получилось **2930** диалогов задачи 1, **2344 в train и 586 в test**. Если числа другие, значит
разбиение отличается от HW1 (проверьте, что `dialog_flags.csv` тот же).

## 3. Старые модели на кросс-валидации

Модель из HW1 уже обучена, а для сравнения нужна оценка на **тех же блоках `folds`**. `cv_metrics` делает `clone`
(пустую копию с теми же параметрами) и 5 раз обучает её на 4 блоках, предсказывая пятый. Вход старых моделей:
тексты всех реплик через пробел (`old_dialog_text`), как в HW1. `kirp_model` считается дольше остальных
(стемминг).

In [ ]:
old_models = su.load_models("hw1_")
X_train_old = [su.old_dialog_text(d) for d in train_dialogs]

for name, model in old_models.items():
    cv_table = su.save_result("cv_table.csv", name, su.cv_metrics(model, X_train_old, y_train, folds))

cv_table.round(3)

## 4. Старые модели на test (контрольная сверка с HW1)

Старые модели вы не подбирали по test, так что смотреть их на test повторно нормально. Числа должны совпасть с таблицей
из HW1 (у `tfidf_logreg` macro-F1 ≈ 0.593, у `kirp_model` ≈ 0.581, у `most_frequent` ≈ 0.252). Если совпало, то всё перенесено верно.

In [ ]:
X_test_old = [su.old_dialog_text(d) for d in test_dialogs]
for name, model in old_models.items():
    test_table = su.save_result("test_table.csv", name, su.all_metrics(y_test, model.predict_proba(X_test_old)))
test_table.round(3)